# MILESTONE M2 — NHIỆM VỤ 5: PHƯƠNG ÁN B — WARD HIERARCHICAL

Notebook thực hiện Task 5 theo đúng các phần 5.0–5.9 của kế hoạch M2. Ward là comparator độc lập của K-Means, nên chỉ thay thuật toán, còn snapshot, universe, tám feature, Robust Scaling và Global K vẫn giữ nguyên.


## Phần 5.0 — Thiết lập file cấu hình Ward

Config Ward V2 được nhân bản từ master config M2 V2. Các rule phương pháp giữ nguyên, chỉ chuyển thuật toán sang `ward` và dùng Global K = 2 đã khóa từ Task 3.


In [1]:
import hashlib
import json
import shutil
import sys
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "configs/experiments/m2_ward_market_only_v2.json").is_file():
            return candidate
    raise RuntimeError("Không tìm thấy repository root hoặc Ward V2 config")

ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

CONFIG_PATH = ROOT / "configs/experiments/m2_ward_market_only_v2.json"
DATA_RUN = ROOT / "artifacts/cafef_primary/cafef-c8-complete-only-v2"
ARTIFACT_DIR = ROOT / "M2/artifacts/m2-task5-ward-v1"
MODELS_DIR = ROOT / "M2/models/ward"

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
FEATURES = config["clustering"]["features"]
GLOBAL_K = config["clustering"]["k"]
MIN_ELIGIBLE = config["min_eligible_count"]

assert config["protocol_scope"] == "m2_market_only_v2"
assert config["stage"] == "M2_TASK_3_GLOBAL_K_FROZEN"
assert config["clustering"]["algorithm"] == "ward"
assert GLOBAL_K == 2

print(f"Protocol: {config['protocol_scope']} | Ward | Global K = {GLOBAL_K}")
print(f"Development: {len(config['development_snapshots'])} snapshot")
print(f"Output artifact: {ARTIFACT_DIR}")


Protocol: m2_market_only_v2 | Ward | Global K = 2
Development: 60 snapshot
Output artifact: D:\WORKPLACE\RAMC\m2-clustering-experiment\M2\artifacts\m2-task5-ward-v1


## Phần 5.1 — Lấy dữ liệu của từng snapshot

Nạp đúng 60 snapshot development từ C8 V2. Mỗi snapshot giữ membership riêng, không lấy membership của một tháng cuối áp ngược về lịch sử.


In [2]:
from delta_t1.experiments.runner import prepare_m2_market_only_snapshots

assert DATA_RUN.joinpath("manifest.json").is_file()
assert DATA_RUN.joinpath("canonical/feature_snapshots.jsonl").is_file()

started = time.time()
source_meta, prepared_snapshots = prepare_m2_market_only_snapshots(DATA_RUN, config)
print(f"Đã nạp {len(prepared_snapshots)} snapshot trong {time.time() - started:.2f}s")
print(f"Data artifact: {source_meta['artifact_id']} | {source_meta['data_version']}")


Đã nạp 60 snapshot trong 6.51s
Data artifact: cafef-c8-complete-only-v2 | CAFEF_C8_COMPLETE_ONLY_V2


## Phần 5.2 — Kiểm tra dữ liệu trước khi phân cụm

Mỗi snapshot phải có ít nhất 100 mã đủ điều kiện, đủ tám feature số hữu hạn, đúng schema và đúng ngày snapshot. Runner fail-closed trước khi trả về dữ liệu cho Ward.


In [3]:
validation = pd.DataFrame([{
    "snapshot_date": item["snapshot_date"],
    "status": item["status"],
    "eligible_count": item["eligible_count"],
    "reason": item["reason"],
} for item in prepared_snapshots])

assert len(validation) == 60
assert (validation["status"] == "ready").all(), "Có snapshot không sẵn sàng; dừng trước khi fit Ward"
assert (validation["eligible_count"] >= MIN_ELIGIBLE).all()

print(f"Universe hợp lệ: {validation.eligible_count.min()} đến {validation.eligible_count.max()} mã/snapshot")
display(validation)


Universe hợp lệ: 116 đến 955 mã/snapshot


,snapshot_date,status,eligible_count,reason
0,2019-01-31,ready,656,None
1,2019-02-28,ready,656,None
2,2019-03-29,ready,662,None
3,2019-04-26,ready,669,None
4,2019-05-31,ready,680,None
5,2019-06-28,ready,695,None
6,2019-07-31,ready,706,None
7,2019-08-30,ready,761,None
8,2019-09-30,ready,781,None
9,2019-10-31,ready,771,None


## Phần 5.3 — Robust Scaling theo snapshot

Mỗi snapshot có median và IQR riêng cho từng feature. Ward nhận vector sau Robust Scaling, nên khác biệt giữa Ward và K-Means chỉ đến từ thuật toán phân cụm.


In [4]:
from delta_t1.features.preprocessing import preprocess

first_snapshot = prepared_snapshots[0]
_, first_scaler = preprocess(first_snapshot["rows"], config["clustering"])
scaler_preview = pd.DataFrame([{
    "feature": feature,
    "median": first_scaler[feature]["center"],
    "IQR": first_scaler[feature]["scale"],
    "lower": first_scaler[feature]["lower"],
    "upper": first_scaler[feature]["upper"],
} for feature in FEATURES])

print(f"Robust scaler tại {first_snapshot['snapshot_date']}")
display(scaler_preview)


Robust scaler tại 2019-01-31


,feature,median,IQR,lower,upper
0,mom_21,0.000000e+00,9.894083e-02,-0.563265,8.715596e-01
1,mom_63,0.000000e+00,1.810996e-01,-0.540232,2.314968e+00
2,mom_126,-1.799230e-03,2.727560e-01,-0.636982,2.840950e+00
3,mom_252,-4.596801e-02,3.871460e-01,-0.800000,4.444702e+00
4,vol_63,4.012176e-01,3.147716e-01,0.000000,1.934825e+00
5,mdd_126,-2.307739e-01,1.800207e-01,-0.702703,0.000000e+00
6,beta_126,2.287831e-01,5.926526e-01,-1.498937,2.068618e+00
7,liquidity_21,3.669881e+07,4.819818e+08,0.000000,1.184559e+11


## Phần 5.4 — Chạy Ward với Global K

Ward chỉ fit với K = 2. Cell này không quét K và không đọc holdout. Với quyền ghi đè đã được phê duyệt, nó chỉ xóa artifact và model của chính Task 5 trước khi tạo kết quả mới.


In [5]:
from delta_t1.clustering.registry import get_algorithm
from delta_t1.experiments.runner import m2_clustering_config

OVERWRITE_EXISTING = True

for target in (ARTIFACT_DIR, MODELS_DIR):
    if target.exists():
        if not OVERWRITE_EXISTING:
            raise FileExistsError(
                f"Output đã tồn tại: {target}. Đặt OVERWRITE_EXISTING = True để chạy lại Task 5."
            )
        shutil.rmtree(target)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
MODELS_DIR.mkdir(parents=True, exist_ok=False)

ward = get_algorithm("ward")
cluster_config = m2_clustering_config(
    config,
    k=GLOBAL_K,
    k_range=[GLOBAL_K],
    momentum_feature="mom_63",
    risk_feature="vol_63",
)

fit_results = []
run_started = time.time()
for prepared in prepared_snapshots:
    result = ward.fit_snapshot(prepared["rows"], cluster_config)
    diagnostic = next(item for item in result["diagnostics"] if item["k"] == GLOBAL_K)
    fit_results.append((prepared, result, diagnostic))

assert len(fit_results) == len(prepared_snapshots)
print(f"Đã fit Ward K={GLOBAL_K} cho {len(fit_results)} snapshot trong {time.time() - run_started:.2f}s")


Đã fit Ward K=2 cho 60 snapshot trong 664.84s


## Phần 5.5 — Lưu cluster assignment

Mỗi mã đủ điều kiện nhận raw cluster label và aligned cluster label. Aligned label được nối giữa các tháng bằng mapping temporal, giúp Task 9 theo dõi chuyển cụm nhất quán.


In [6]:
from delta_t1.evaluation.temporal_metrics import compare

run_id = "m2-task5-ward-v1"
assignments = []
aligned_by_snapshot = {}
previous_result, previous_aligned = None, None

for prepared, result, _ in fit_results:
    snapshot_date = prepared["snapshot_date"]
    if previous_result is not None:
        mapping = compare(previous_result, result).get("mapping")
        if mapping is not None:
            current_aligned = {
                current: previous_aligned[prior]
                for current, prior in mapping.items()
            }
        else:
            current_aligned = None
    else:
        current_aligned = None

    if current_aligned is None:
        current_aligned = {
            profile["raw_cluster_id"]: profile["economic_rank"]
            for profile in result["profiles"]
        }

    aligned_by_snapshot[snapshot_date] = current_aligned
    for row, raw_cluster_id in zip(result["rows"], result["labels"]):
        assignments.append({
            "run_id": run_id,
            "snapshot_date": snapshot_date,
            "security_id": row["security_id"],
            "raw_cluster_id": raw_cluster_id,
            "aligned_cluster_id": current_aligned[raw_cluster_id],
            "data_version": source_meta["data_version"],
        })

    previous_result, previous_aligned = result, current_aligned

assignments_frame = pd.DataFrame(assignments)
assert len(assignments_frame) == int(validation["eligible_count"].sum())
display(assignments_frame.head())


,run_id,snapshot_date,security_id,raw_cluster_id,aligned_cluster_id,data_version
0,m2-task5-ward-v1,2019-01-31,KBS:HNX:ADC,0,0,CAFEF_C8_COMPLETE_ONLY_V2
1,m2-task5-ward-v1,2019-01-31,KBS:HNX:ALT,0,0,CAFEF_C8_COMPLETE_ONLY_V2
2,m2-task5-ward-v1,2019-01-31,KBS:HNX:AMC,0,0,CAFEF_C8_COMPLETE_ONLY_V2
3,m2-task5-ward-v1,2019-01-31,KBS:HNX:AME,0,0,CAFEF_C8_COMPLETE_ONLY_V2
4,m2-task5-ward-v1,2019-01-31,KBS:HNX:AMV,0,0,CAFEF_C8_COMPLETE_ONLY_V2


## Phần 5.6 — Xây cluster representative và cluster profile

Ward không tối ưu centroid như K-Means. Sau khi gán cụm, notebook tính representative bằng mean của thành viên trong cụm theo cùng quy tắc profile, cùng tám feature và schema 13 cột của M2.


In [7]:
profiles = []
scaler_rows = []

for prepared, result, _ in fit_results:
    snapshot_date = prepared["snapshot_date"]
    aligned_ids = aligned_by_snapshot[snapshot_date]

    for profile in result["profiles"]:
        profiles.append({
            "snapshot_date": snapshot_date,
            "raw_cluster_id": profile["raw_cluster_id"],
            "aligned_cluster_id": aligned_ids[profile["raw_cluster_id"]],
            "size": profile["size"],
            "size_ratio": profile["size"] / prepared["eligible_count"],
            **{feature: profile["centroid"][feature] for feature in FEATURES},
        })

    for feature in FEATURES:
        scaler_rows.append({
            "snapshot_date": snapshot_date,
            "feature": feature,
            **result["model"]["scaler"][feature],
        })

profiles_frame = pd.DataFrame(profiles)
scalers_frame = pd.DataFrame(scaler_rows)

assert list(profiles_frame.columns) == [
    "snapshot_date", "raw_cluster_id", "aligned_cluster_id",
    "size", "size_ratio", *FEATURES,
]
assert len(profiles_frame) == len(prepared_snapshots) * GLOBAL_K
assert not profiles_frame.isna().any().any()

display(profiles_frame.head())


,snapshot_date,raw_cluster_id,aligned_cluster_id,size,size_ratio,mom_21,mom_63,mom_126,mom_252,vol_63,mdd_126,beta_126,liquidity_21
0,2019-01-31,0,0,648,0.987805,0.003277,0.029398,0.048524,0.010901,0.465075,-0.253157,0.287043,1.969936e+09
1,2019-01-31,1,1,8,0.012195,0.039462,-0.007194,-0.045527,-0.178291,0.279032,-0.232461,1.139304,1.062323e+11
2,2019-02-28,0,0,651,0.992378,0.032227,0.045305,0.058408,0.039011,0.458351,-0.250515,0.278392,3.182627e+09
3,2019-02-28,1,1,5,0.007622,0.077639,0.025520,-0.072956,-0.189018,0.301457,-0.270606,1.186922,1.725316e+11
4,2019-03-29,0,0,627,0.947130,0.036937,0.062473,0.062577,0.058086,0.482040,-0.250775,0.255788,1.572620e+09


## Phần 5.7 — Tính quality metrics

Ward được đánh giá tại từng snapshot bằng Silhouette, Davies–Bouldin, Calinski–Harabasz và Cluster Balance. Inertia được lưu như chẩn đoán phụ, không dùng để chọn lại K.


In [8]:
diagnostics = []
for prepared, _, diagnostic in fit_results:
    diagnostics.append({
        "snapshot_date": prepared["snapshot_date"],
        **diagnostic,
    })

diagnostics_frame = pd.DataFrame(diagnostics)
assert len(diagnostics_frame) == len(prepared_snapshots)

quality_columns = [
    "snapshot_date", "silhouette", "davies_bouldin",
    "calinski_harabasz", "inertia", "cluster_balance",
]
display(diagnostics_frame[quality_columns])

quality_median = diagnostics_frame[
    ["silhouette", "davies_bouldin", "calinski_harabasz", "cluster_balance"]
].median().rename("median").to_frame()
display(quality_median)


,snapshot_date,silhouette,davies_bouldin,calinski_harabasz,inertia,cluster_balance
0,2019-01-31,0.949454,0.096075,2060.176839,1.173967e+05,0.012346
1,2019-02-28,0.949498,0.178409,1084.828444,2.510187e+05,0.007680
2,2019-03-29,0.883656,0.491123,1407.104752,6.315804e+04,0.055821
3,2019-04-26,0.885047,0.533150,1093.885643,9.144885e+04,0.048589
4,2019-05-31,0.972832,0.018141,741.963922,2.118663e+05,0.001473
5,2019-06-28,0.931426,0.372667,1907.928316,1.315401e+05,0.022059
6,2019-07-31,0.942458,0.265989,879.259764,2.223972e+05,0.007133
7,2019-08-30,0.944814,0.265593,1206.069632,1.779560e+05,0.007947
8,2019-09-30,0.899473,0.501253,1503.615053,1.986217e+05,0.048322
9,2019-10-31,0.903065,0.500279,1096.703828,2.348767e+05,0.033512


,median
silhouette,0.933990
davies_bouldin,0.317552
calinski_harabasz,1020.404838
cluster_balance,0.018855


## Phần 5.8 — Lặp lại cho toàn bộ development

Các bước lọc universe, kiểm tra feature, Robust Scaling, Ward K = 2, assignment, profile và metrics đã được thực hiện độc lập cho đủ 60 snapshot development.


In [9]:
coverage = pd.DataFrame([{
    "snapshot_date": prepared["snapshot_date"],
    "eligible_count": prepared["eligible_count"],
    "assignments": sum(
        assignments_frame["snapshot_date"] == prepared["snapshot_date"]
    ),
    "profiles": sum(
        profiles_frame["snapshot_date"] == prepared["snapshot_date"]
    ),
    "diagnostics": sum(
        diagnostics_frame["snapshot_date"] == prepared["snapshot_date"]
    ),
} for prepared, _, _ in fit_results])

assert (coverage["assignments"] == coverage["eligible_count"]).all()
assert (coverage["profiles"] == GLOBAL_K).all()
assert (coverage["diagnostics"] == 1).all()

print("Coverage 60 snapshot development: PASS")
display(coverage)


Coverage 60 snapshot development: PASS


,snapshot_date,eligible_count,assignments,profiles,diagnostics
0,2019-01-31,656,656,2,1
1,2019-02-28,656,656,2,1
2,2019-03-29,662,662,2,1
3,2019-04-26,669,669,2,1
4,2019-05-31,680,680,2,1
5,2019-06-28,695,695,2,1
6,2019-07-31,706,706,2,1
7,2019-08-30,761,761,2,1
8,2019-09-30,781,781,2,1
9,2019-10-31,771,771,2,1


## Phần 5.9 — Lưu artifact Ward

Xuất artifact và model đã fit để Task 7 đọc quality metrics, Task 8 đọc cluster profiles, Task 9 đọc assignment/profile/scaler và Task 10 so sánh Ward với các phương án khác.


In [10]:
def write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


# Xuất model Ward theo snapshot: scaler, tâm cụm scaled, linkage và metadata fit.
linkage_records = []
universe_records = []
representatives = []
cluster_sizes = []
model_index = []

for prepared, result, diagnostic in fit_results:
    snapshot_date = prepared["snapshot_date"]
    model = result["model"]
    linkage = model.get("linkage")
    assert isinstance(linkage, list)
    assert len(linkage) == prepared["eligible_count"] - GLOBAL_K

    model_path = MODELS_DIR / f"{snapshot_date}.json"
    model_document = {
        "run_id": run_id,
        "snapshot_date": snapshot_date,
        "algorithm": "ward",
        "global_k": GLOBAL_K,
        "feature_order": FEATURES,
        "eligible_count": prepared["eligible_count"],
        "model": model,
        "diagnostics": diagnostic,
    }
    model_path.write_text(
        json.dumps(model_document, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    model_index.append({
        "snapshot_date": snapshot_date,
        "model_path": str(model_path.relative_to(ROOT)).replace("\\", "/"),
        "linkage_merge_count": len(linkage),
        "sha256": sha256(model_path),
    })

    for merge_index, merge in enumerate(linkage):
        linkage_records.append({
            "snapshot_date": snapshot_date,
            "merge_index": merge_index,
            **merge,
        })

    aligned_ids = aligned_by_snapshot[snapshot_date]
    for profile in result["profiles"]:
        representative = {
            "snapshot_date": snapshot_date,
            "raw_cluster_id": profile["raw_cluster_id"],
            "aligned_cluster_id": aligned_ids[profile["raw_cluster_id"]],
            "semantic_label": profile["semantic_label"],
            "economic_rank": profile["economic_rank"],
            "size": profile["size"],
            "size_ratio": profile["size"] / prepared["eligible_count"],
            **{feature: profile["centroid"][feature] for feature in FEATURES},
        }
        representatives.append(representative)
        cluster_sizes.append({
            "snapshot_date": snapshot_date,
            "raw_cluster_id": representative["raw_cluster_id"],
            "aligned_cluster_id": representative["aligned_cluster_id"],
            "size": representative["size"],
            "size_ratio": representative["size_ratio"],
        })

    for row in prepared["rows"]:
        universe_records.append({
            "snapshot_date": snapshot_date,
            "security_id": row["security_id"],
            "ticker": row.get("ticker", row["security_id"].split(":")[-1]),
            "eligibility_field": config["eligibility_field"],
            "eligible": True,
        })

assert len(universe_records) == len(assignments)
assert len(linkage_records) == int(validation["eligible_count"].sum()) - len(prepared_snapshots) * GLOBAL_K
assert len(representatives) == len(profiles_frame)
assert len(cluster_sizes) == len(profiles_frame)

representatives_frame = pd.DataFrame(representatives)
cluster_sizes_frame = pd.DataFrame(cluster_sizes)
universe_frame = pd.DataFrame(universe_records)

# JSONL là artifact chuẩn; CSV là bản thuận tiện cho kiểm tra, Task 7 và Task 8.
write_jsonl(ARTIFACT_DIR / "assignments.jsonl", assignments)
write_jsonl(ARTIFACT_DIR / "profiles.jsonl", profiles)
write_jsonl(ARTIFACT_DIR / "diagnostics.jsonl", diagnostics)
write_jsonl(ARTIFACT_DIR / "eligible_universe.jsonl", universe_records)
write_jsonl(ARTIFACT_DIR / "cluster_representatives.jsonl", representatives)
write_jsonl(ARTIFACT_DIR / "cluster_sizes.jsonl", cluster_sizes)
write_jsonl(ARTIFACT_DIR / "linkages.jsonl", linkage_records)
write_jsonl(ARTIFACT_DIR / "models_manifest.jsonl", model_index)
write_jsonl(ARTIFACT_DIR / "skipped_snapshots.jsonl", [])

assignments_frame.to_csv(ARTIFACT_DIR / "assignments.csv", index=False, encoding="utf-8")
profiles_frame.to_csv(ARTIFACT_DIR / "profiles.csv", index=False, encoding="utf-8")
profiles_frame.to_csv(ARTIFACT_DIR / "cluster_profiles.csv", index=False, encoding="utf-8")
representatives_frame.to_csv(ARTIFACT_DIR / "cluster_representatives.csv", index=False, encoding="utf-8")
cluster_sizes_frame.to_csv(ARTIFACT_DIR / "cluster_sizes.csv", index=False, encoding="utf-8")
universe_frame.to_csv(ARTIFACT_DIR / "eligible_universe.csv", index=False, encoding="utf-8")
pd.DataFrame(linkage_records).to_csv(ARTIFACT_DIR / "linkages.csv", index=False, encoding="utf-8")

# diagnostics.csv có đúng schema các Task 7 và 10 đọc trực tiếp.
diagnostics_csv = diagnostics_frame.copy()
diagnostics_csv["cluster_sizes"] = diagnostics_csv["cluster_sizes"].map(
    lambda value: json.dumps(value, sort_keys=True)
)
diagnostics_csv.to_csv(ARTIFACT_DIR / "diagnostics.csv", index=False, encoding="utf-8")
scalers_frame.to_csv(ARTIFACT_DIR / "scaler_parameters.csv", index=False, encoding="utf-8")
validation.to_csv(ARTIFACT_DIR / "snapshot_validation.csv", index=False, encoding="utf-8")
coverage.to_csv(ARTIFACT_DIR / "coverage.csv", index=False, encoding="utf-8")

# Đóng băng config và nguồn input ngay trong artifact để audit có thể chạy độc lập.
(ARTIFACT_DIR / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
(ARTIFACT_DIR / "source_manifest.json").write_text(
    (DATA_RUN / "manifest.json").read_text(encoding="utf-8"), encoding="utf-8"
)
run_metadata = {
    "run_id": run_id,
    "stage": "M2_TASK_5_WARD_DEVELOPMENT",
    "status": "complete",
    "algorithm": "ward",
    "global_k": GLOBAL_K,
    "feature_set": FEATURES,
    "eligibility_field": config["eligibility_field"],
    "minimum_eligible_count": MIN_ELIGIBLE,
    "preprocessing": config["clustering"]["scaling"],
    "scaler_fit_scope": config["clustering"]["scaler_fit_scope"],
    "development_snapshots": config["development_snapshots"],
    "data_artifact_id": source_meta["artifact_id"],
    "data_version": source_meta["data_version"],
    "linkage_schema": ["left", "right", "distance", "size"],
    "created_at_utc": datetime.utcnow().replace(microsecond=0).isoformat() + "Z",
}
(ARTIFACT_DIR / "run_metadata.json").write_text(
    json.dumps(run_metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

artifact_files = [path for path in sorted(ARTIFACT_DIR.iterdir()) if path.is_file()]
manifest = {
    **run_metadata,
    "development_snapshot_count": len(prepared_snapshots),
    "assignment_count": len(assignments),
    "eligible_universe_count": len(universe_records),
    "cluster_representative_count": len(representatives),
    "linkage_merge_count": len(linkage_records),
    "config_sha256": sha256(CONFIG_PATH),
    "source_manifest_sha256": sha256(DATA_RUN / "manifest.json"),
    "feature_snapshot_sha256": sha256(DATA_RUN / "canonical/feature_snapshots.jsonl"),
    "outputs": {path.name: sha256(path) for path in artifact_files},
    "models": {item["model_path"]: item["sha256"] for item in model_index},
}
(ARTIFACT_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

expected_files = {
    "assignments.jsonl", "profiles.jsonl", "diagnostics.jsonl", "manifest.json",
    "eligible_universe.jsonl", "cluster_representatives.jsonl", "cluster_sizes.jsonl",
    "linkages.jsonl", "config.json", "source_manifest.json", "run_metadata.json",
    "assignments.csv", "cluster_profiles.csv", "diagnostics.csv", "scaler_parameters.csv",
}
assert expected_files <= {path.name for path in ARTIFACT_DIR.iterdir()}
assert len(model_index) == len(prepared_snapshots)

print(f"Artifact Ward: {ARTIFACT_DIR}")
print(f"Model Ward: {MODELS_DIR}")
print(f"Assignments / universe: {len(assignments_frame):,} / {len(universe_frame):,}")
print(f"Representatives / linkage merges: {len(representatives_frame)} / {len(linkage_records):,}")
print("Audit output contract: PASS")

C:\Users\phuon\AppData\Local\Temp\ipykernel_8696\981307296.py:144: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at_utc": datetime.utcnow().replace(microsecond=0).isoformat() + "Z",


Artifact Ward: D:\WORKPLACE\RAMC\m2-clustering-experiment\M2\artifacts\m2-task5-ward-v1
Model Ward: D:\WORKPLACE\RAMC\m2-clustering-experiment\M2\models\ward
Assignments / universe: 29,907 / 29,907
Representatives / linkage merges: 120 / 29,787
Audit output contract: PASS


## Bàn giao

Task 7 dùng `diagnostics.csv`. Task 8 dùng `cluster_profiles.csv`. Task 9 dùng assignment, profile và scaler. Task 5 không mở holdout.
